# Notebook 01: Exploring MIMIC-IV and MIMIC-IV-Note

**Author:** Anthony Amit Biswas

## What this notebook does

Explores the MIMIC-IV tables and ICU discharge summaries, validates their relationships, and constructs the cohort used by the downstream clinical NLP workflow.


# **1. Import Required Libraries**

In [ ]:
# Importing Required Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 500)

print("Libraries imported successfully.")

# **2. Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **3. Definie Dataset Locations**

In [ ]:
# Setting up Dataset Paths

DATA_PATH = "/content/drive/MyDrive/Dissertation/data/"

DISCHARGE_FILE = DATA_PATH + "discharge.csv.gz"
PATIENTS_FILE = DATA_PATH + "patients.csv.gz"
ADMISSIONS_FILE = DATA_PATH + "admissions.csv.gz"
ICUSTAYS_FILE = DATA_PATH + "icustays.csv.gz"

In [ ]:
# Loading Sample Data

discharge_sample = pd.read_csv(DISCHARGE_FILE, compression="gzip", nrows=5)

patients_sample = pd.read_csv(PATIENTS_FILE, compression="gzip", nrows=5)

admissions_sample = pd.read_csv(ADMISSIONS_FILE, compression="gzip", nrows=5)

icustays_sample = pd.read_csv(ICUSTAYS_FILE, compression="gzip", nrows=5)

print("Datasets loaded successfully.")

In [ ]:
#Displaying samples

print("DISCHARGE")
display(discharge_sample)

print("PATIENTS")
display(patients_sample)

print("ADMISSIONS")
display(admissions_sample)

print("ICU STAYS")
display(icustays_sample)

##2. Loading the Complete MIMIC-IV Datasets

* The sample rows confirmed that the files are accessible and that the main identifiers are available.

* The complete structured datasets are now loaded. These tables are relatively small compared with the discharge-note file.

* The discharge-note dataset is also loaded in full because the text column is required for the clinical NLP analysis.




# **4. Load MIMIC-IV Datasets**

**Loading the complete structured datasets**

In [ ]:
# Loading the complete structured MIMIC-IV datasets
#
# The patient table contains demographic information.
# The admissions table contains hospital admission details.
# The ICU stays table contains information about each ICU stay.
#
# Date columns are converted into datetime format during loading
# so that they can be analysed correctly later.


patients = pd.read_csv(
    PATIENTS_FILE,
    compression="gzip",
    parse_dates=["dod"]
)

admissions = pd.read_csv(
    ADMISSIONS_FILE,
    compression="gzip",
    parse_dates=[
        "admittime",
        "dischtime",
        "deathtime",
        "edregtime",
        "edouttime"
    ]
)

icustays = pd.read_csv(
    ICUSTAYS_FILE,
    compression="gzip",
    parse_dates=[
        "intime",
        "outtime"
    ]
)

print("-" * 70)
print("STRUCTURED DATASETS LOADED SUCCESSFULLY")
print("-" * 70)

print(f"Patients dataset shape   : {patients.shape}")
print(f"Admissions dataset shape : {admissions.shape}")
print(f"ICU stays dataset shape  : {icustays.shape}")

**Loading the complete discharge-note dataset**

In [ ]:
# Loading the complete MIMIC-IV discharge-note dataset
#
# This is the main text dataset used in the dissertation.
# It contains the discharge summaries that will later be used
# for clinical NLP and entity extraction.
#
# The file is large, so loading may take several minutes.
# The compressed .csv.gz file can be read directly by pandas.

discharge = pd.read_csv(
    DISCHARGE_FILE,
    compression="gzip",
    parse_dates=[
        "charttime",
        "storetime"
    ],
    low_memory=False
)

print("-" * 70)
print("DISCHARGE DATASET LOADED SUCCESSFULLY")
print("-" * 70)

print(f"Discharge dataset shape : {discharge.shape}")

**Checking dataset dimensions and columns**

In [ ]:
# Reviewing the number of rows, columns and column names
#
# This confirms the size and structure of each complete dataset
# before any filtering, merging or text analysis is performed.

datasets = {
    "Patients": patients,
    "Admissions": admissions,
    "ICU Stays": icustays,
    "Discharge Notes": discharge
}

for dataset_name, dataframe in datasets.items():

    print("\n" + "-" * 70)
    print(dataset_name.upper())
    print("-" * 70)

    print(f"Number of rows    : {dataframe.shape[0]:,}")
    print(f"Number of columns : {dataframe.shape[1]}")
    print("Column names:")
    print(dataframe.columns.tolist())

**Checking missing values and duplicate rows**

In [ ]:
# Here, Performing an initial data-quality assessment
#
# Missing values and duplicated rows may affect later analysis.
# This step provides a simple overview of the quality of each
# dataset before the ICU cohort is constructed.

for dataset_name, dataframe in datasets.items():

    print("\n" + "-" * 70)
    print(f"DATA QUALITY CHECK: {dataset_name.upper()}")
    print("-" * 70)

    print(f"Duplicate rows : {dataframe.duplicated().sum():,}")

    missing_values = dataframe.isnull().sum()
    missing_values = missing_values[missing_values > 0]

    if len(missing_values) == 0:
        print("Missing values : None")
    else:
        print("Missing values by column:")
        print(missing_values)

**Confirming the key identifiers**

In [ ]:
# Confirm the identifiers used to connect the datasets
#
# subject_id identifies the patient.
# hadm_id identifies the hospital admission.
# stay_id identifies the ICU stay.
# note_id identifies the individual discharge note.

print("-" * 70)
print("KEY IDENTIFIERS")
print("-" * 70)

print("Patients table:")
print(" - subject_id")

print("\nAdmissions table:")
print(" - subject_id")
print(" - hadm_id")

print("\nICU stays table:")
print(" - subject_id")
print(" - hadm_id")
print(" - stay_id")

print("\nDischarge-note table:")
print(" - note_id")
print(" - subject_id")
print(" - hadm_id")

**Identifying hospital admissions that included an ICU stay**

In [ ]:
# Identifying the unique hospital admissions involving ICU care
#
# The discharge-note dataset contains summaries for hospital
# admissions in general. Since this dissertation focuses on ICU
# patients, I first identify the hospital admissions that appear
# in the ICU stays table.
#
# hadm_id is used because it connects the ICU stay with the
# corresponding hospital admission and discharge summary.


icu_admission_ids = icustays["hadm_id"].dropna().unique()

print("-" * 70)
print("ICU ADMISSION IDENTIFICATION")
print("-" * 70)

print(
    f"Number of unique hospital admissions with an ICU stay: "
    f"{len(icu_admission_ids):,}"
)

**Creating the ICU discharge-summary cohort**

In [ ]:
# Filtering the discharge summaries to ICU-related admissions
#
# Only discharge summaries whose hadm_id appears in the ICU
# stays table are retained.
#
# This creates the main ICU discharge-summary cohort that will
# later be used for clinical NLP analysis.

icu_discharge = discharge[
    discharge["hadm_id"].isin(icu_admission_ids)
].copy()

print("-" * 70)
print("ICU DISCHARGE-SUMMARY COHORT")
print("-" * 70)

print(f"All discharge summaries          : {len(discharge):,}")
print(f"ICU-associated discharge summaries: {len(icu_discharge):,}")

icu_percentage = (
    len(icu_discharge) / len(discharge) * 100
)

print(
    f"Percentage of discharge summaries linked to ICU admissions: "
    f"{icu_percentage:.2f}%"
)

**Checking whether identifiers are missing in the ICU cohort**

In [ ]:
# Check the ICU discharge cohort for missing identifiers
#
# subject_id and hadm_id are required for linking the notes
# with patient, admission and ICU information.

print("-" * 70)
print("ICU COHORT IDENTIFIER CHECK")
print("-" * 70)

print(
    "Missing subject_id values:",
    icu_discharge["subject_id"].isnull().sum()
)

print(
    "Missing hadm_id values:",
    icu_discharge["hadm_id"].isnull().sum()
)

print(
    "Missing note_id values:",
    icu_discharge["note_id"].isnull().sum()
)

**Merging ICU discharge notes with ICU stay information**

In [ ]:
# Linking up ICU discharge summaries with ICU stay information
#
# The merge uses both subject_id and hadm_id to reduce the risk
# of incorrect matches.
#
# One hospital admission may contain more than one ICU stay.
# Therefore, one discharge summary may appear more than once
# after the merge if the admission included multiple ICU stays.

icu_discharge_with_stays = icu_discharge.merge(
    icustays[
        [
            "subject_id",
            "hadm_id",
            "stay_id",
            "first_careunit",
            "last_careunit",
            "intime",
            "outtime",
            "los"
        ]
    ],
    on=[
        "subject_id",
        "hadm_id"
    ],
    how="left"
)

print("-" * 70)
print("ICU DISCHARGE NOTES MERGED WITH ICU STAYS")
print("-" * 70)

print(
    f"Rows before merge : {len(icu_discharge):,}"
)

print(
    f"Rows after merge  : {len(icu_discharge_with_stays):,}"
)

**Displaying the merged sample**

In [ ]:
# Displaying a sample of the linked ICU discharge-summary data
#
# This confirms that the discharge notes have been connected
# successfully with the correct ICU stay information.

display(
    icu_discharge_with_stays[
        [
            "note_id",
            "subject_id",
            "hadm_id",
            "stay_id",
            "first_careunit",
            "last_careunit",
            "los"
        ]
    ].head(10)
)

## 3. Constructing a Clean ICU Discharge-Summary Cohort

Some hospital admissions contain multiple ICU stays. Therefore, directly merging the discharge summaries with the ICU stay table can create multiple rows for the same discharge note.

For the main NLP analysis, each discharge summary should appear only once. ICU-stay information will therefore be aggregated at the hospital-admission level before it is merged with the discharge-note dataset.

**Examine admissions with multiple ICU stays**

In [ ]:
# Examine the number of ICU stays within each hospital admission
#
# A hospital admission may contain one or more ICU stays.
# This explains why the number of rows increased after merging
# the discharge summaries with the ICU stay table.

icu_stays_per_admission = (
    icustays.groupby("hadm_id")
    .size()
    .reset_index(name="number_of_icu_stays")
)

print("-" * 70)
print("ICU STAYS PER HOSPITAL ADMISSION")
print("-" * 70)

print(
    f"Admissions with exactly one ICU stay: "
    f"{(icu_stays_per_admission['number_of_icu_stays'] == 1).sum():,}"
)

print(
    f"Admissions with more than one ICU stay: "
    f"{(icu_stays_per_admission['number_of_icu_stays'] > 1).sum():,}"
)

print(
    f"Maximum number of ICU stays in one hospital admission: "
    f"{icu_stays_per_admission['number_of_icu_stays'].max():,}"
)

display(
    icu_stays_per_admission
    .sort_values("number_of_icu_stays", ascending=False)
    .head(10)
)

**Aggregating ICU stay information by hospital admission**

In [ ]:
# Aggregating ICU information at hospital-admission level
#
# The main NLP unit is the discharge summary rather than the
# individual ICU stay. Therefore, ICU information is summarised
# for each hospital admission before it is merged with the notes.
#
# The aggregated table contains:
# - the number of ICU stays,
# - the first ICU admission time,
# - the final ICU discharge time,
# - the total recorded ICU length of stay,
# - all first care units involved,
# - all last care units involved.


icu_admission_summary = (
    icustays.groupby(
        ["subject_id", "hadm_id"],
        as_index=False
    )
    .agg(
        number_of_icu_stays=("stay_id", "nunique"),
        first_icu_intime=("intime", "min"),
        final_icu_outtime=("outtime", "max"),
        total_icu_los_days=("los", "sum"),
        first_careunits=(
            "first_careunit",
            lambda values: " | ".join(
                sorted(values.dropna().astype(str).unique())
            )
        ),
        last_careunits=(
            "last_careunit",
            lambda values: " | ".join(
                sorted(values.dropna().astype(str).unique())
            )
        )
    )
)

print("-" * 70)
print("AGGREGATED ICU ADMISSION TABLE")
print("-" * 70)

print(
    f"Rows in aggregated ICU admission table: "
    f"{len(icu_admission_summary):,}"
)

display(icu_admission_summary.head())

**Merging the discharge summaries with aggregated ICU information**

In [ ]:
# Creating one clean row for each ICU discharge summary
#
# The discharge-summary cohort is merged with the aggregated ICU
# admission table using subject_id and hadm_id.
#
# Since the ICU table now contains one row per hospital admission,
# the number of discharge notes should not increase after merging.

icu_notes = icu_discharge.merge(
    icu_admission_summary,
    on=["subject_id", "hadm_id"],
    how="left",
    validate="many_to_one"
)

print("-" * 70)
print("CLEAN ICU DISCHARGE-SUMMARY DATASET")
print("-" * 70)

print(f"Rows before merge : {len(icu_discharge):,}")
print(f"Rows after merge  : {len(icu_notes):,}")
print(f"Unique note IDs   : {icu_notes['note_id'].nunique():,}")
print(f"Duplicate note IDs: {icu_notes['note_id'].duplicated().sum():,}")

**Adding hospital admission information**

In [ ]:
# Adding hospital admission information to the ICU note cohort
#
# Admission details provide useful contextual information such as:
# - hospital admission and discharge times,
# - admission type,
# - admission and discharge locations,
# - in-hospital mortality.
#
# hadm_id should uniquely identify each hospital admission.

admission_columns = [
    "subject_id",
    "hadm_id",
    "admittime",
    "dischtime",
    "deathtime",
    "admission_type",
    "admission_location",
    "discharge_location",
    "insurance",
    "language",
    "marital_status",
    "race",
    "hospital_expire_flag"
]

icu_notes = icu_notes.merge(
    admissions[admission_columns],
    on=["subject_id", "hadm_id"],
    how="left",
    validate="many_to_one"
)

print("-" * 70)
print("HOSPITAL ADMISSION INFORMATION ADDED")
print("-" * 70)

print(f"Rows in ICU note cohort: {len(icu_notes):,}")

print(
    "Rows without matching admission information:",
    icu_notes["admittime"].isnull().sum()
)

**Adding patient demographic information**

In [ ]:
# Adding patient demographic information
#
# The patient table is linked using subject_id.
#
# anchor_age is the patient's age in the anchor year.
# Because MIMIC applies date shifting for de-identification,
# age at admission is estimated using the difference between
# the admission year and anchor year.

patient_columns = [
    "subject_id",
    "gender",
    "anchor_age",
    "anchor_year",
    "anchor_year_group",
    "dod"
]

icu_notes = icu_notes.merge(
    patients[patient_columns],
    on="subject_id",
    how="left",
    validate="many_to_one"
)

print("-" * 70)
print("PATIENT INFORMATION ADDED")
print("-" * 70)

print(f"Rows in final linked cohort: {len(icu_notes):,}")

print(
    "Rows without matching patient information:",
    icu_notes["gender"].isnull().sum()
)

**Calculating estimated age and hospital length of stay**

In [ ]:
# Deriving additional variables for descriptive analysis
#
# Estimated age at admission is calculated using anchor_age,
# anchor_year and the year of hospital admission.
#
# Hospital length of stay is calculated as the difference between
# hospital discharge time and hospital admission time.

icu_notes["estimated_age_at_admission"] = (
    icu_notes["anchor_age"]
    + icu_notes["admittime"].dt.year
    - icu_notes["anchor_year"]
)

icu_notes["hospital_los_days"] = (
    icu_notes["dischtime"] - icu_notes["admittime"]
).dt.total_seconds() / (24 * 60 * 60)

print("-" * 70)
print("DERIVED VARIABLES CREATED")
print("-" * 70)

print(
    "Estimated age range:",
    icu_notes["estimated_age_at_admission"].min(),
    "to",
    icu_notes["estimated_age_at_admission"].max()
)

print(
    "Median hospital length of stay:",
    round(icu_notes["hospital_los_days"].median(), 2),
    "days"
)

print(
    "Median ICU length of stay:",
    round(icu_notes["total_icu_los_days"].median(), 2),
    "days"
)

**Checking the final cohort structure**

In [ ]:
# Reviewing the structure of the final linked ICU note cohort
#
# This confirms that the discharge text, ICU information,
# hospital admission details and patient information are all
# available within one analysis table.

print("-" * 70)
print("FINAL ICU NOTE COHORT STRUCTURE")
print("-" * 70)

print(f"Number of rows    : {icu_notes.shape[0]:,}")
print(f"Number of columns : {icu_notes.shape[1]}")
print(f"Unique patients   : {icu_notes['subject_id'].nunique():,}")
print(f"Unique admissions : {icu_notes['hadm_id'].nunique():,}")
print(f"Unique notes      : {icu_notes['note_id'].nunique():,}")

print("\nColumn names:")
print(icu_notes.columns.tolist())

## 4. Exploratory Analysis of the ICU Discharge Summaries

The following analysis examines the size, length and basic characteristics of the ICU discharge-summary cohort. These results help determine the computational requirements for the clinical NLP pipeline and provide descriptive statistics for the dissertation and research poster.

**Examine note types and note sequences**

In [ ]:
# Examine note types and note sequence numbers
#
# MIMIC-IV-Note may contain more than one version of a discharge
# summary for the same hospital admission.
#
# note_seq indicates the sequence of notes written for a patient,
# while note_type identifies the type of clinical note.

print("-" * 70)
print("NOTE TYPE DISTRIBUTION")
print("-" * 70)

print(icu_notes["note_type"].value_counts(dropna=False))

print("\n" + "-" * 70)
print("NUMBER OF DISCHARGE NOTES PER HOSPITAL ADMISSION")
print("-" * 70)

notes_per_admission = (
    icu_notes.groupby("hadm_id")["note_id"]
    .nunique()
)

print(
    f"Admissions with one discharge note: "
    f"{(notes_per_admission == 1).sum():,}"
)

print(
    f"Admissions with more than one discharge note: "
    f"{(notes_per_admission > 1).sum():,}"
)

print(
    f"Maximum discharge notes for one admission: "
    f"{notes_per_admission.max():,}"
)

**Calculating discharge-summary lengths**

In [ ]:
# Calculating the length of each discharge summary
#
# Character count measures the total number of characters.
# Word count provides a more interpretable measure of note length.
#
# These values help estimate the amount of text that the NLP
# toolkits will need to process.

icu_notes["character_count"] = (
    icu_notes["text"]
    .fillna("")
    .str.len()
)

icu_notes["word_count"] = (
    icu_notes["text"]
    .fillna("")
    .str.split()
    .str.len()
)

print("-" * 70)
print("DISCHARGE-SUMMARY LENGTH STATISTICS")
print("-" * 70)

note_length_statistics = (
    icu_notes[
        [
            "character_count",
            "word_count"
        ]
    ]
    .describe()
    .round(2)
)

display(note_length_statistics)

**Displaying key note-length statistics**

In [ ]:
# Presenting the main discharge-summary length statistics
#
# Median values are useful because clinical note lengths may be
# highly skewed by a small number of very long summaries.

print("-" * 70)
print("KEY NOTE-LENGTH RESULTS")
print("-" * 70)

print(
    f"Median word count: "
    f"{icu_notes['word_count'].median():,.0f}"
)

print(
    f"Mean word count: "
    f"{icu_notes['word_count'].mean():,.2f}"
)

print(
    f"Shortest note: "
    f"{icu_notes['word_count'].min():,.0f} words"
)

print(
    f"Longest note: "
    f"{icu_notes['word_count'].max():,.0f} words"
)

print(
    f"Notes longer than 5,000 words: "
    f"{(icu_notes['word_count'] > 5000).sum():,}"
)

**Ploting the note-length distribution**

In [ ]:
# Visualising the distribution of discharge-summary word counts
#
# The upper 1% of note lengths is excluded from the plot so that
# the main distribution remains visible.
#
# The complete dataset is not modified by this visualisation.

word_count_limit = icu_notes["word_count"].quantile(0.99)

plt.figure(figsize=(10, 6))

plt.hist(
    icu_notes.loc[
        icu_notes["word_count"] <= word_count_limit,
        "word_count"
    ],
    bins=50
)

plt.title(
    "Distribution of Word Counts in ICU Discharge Summaries"
)

plt.xlabel("Number of Words")
plt.ylabel("Number of Discharge Summaries")

plt.tight_layout()
plt.show()

**Examine ICU care-unit distribution**

In [ ]:
# Examine the distribution of ICU care units
#
# This shows the types of intensive care environments represented
# in the selected discharge-summary cohort.
#
# Because an admission may involve more than one care unit,
# combined care-unit labels may appear in the aggregated table.

careunit_distribution = (
    icu_notes["first_careunits"]
    .value_counts()
    .head(15)
)

print("-" * 70)
print("MOST COMMON ICU CARE-UNIT COMBINATIONS")
print("-" * 70)

display(
    careunit_distribution
    .rename_axis("ICU care unit")
    .reset_index(name="number_of_discharge_summaries")
)

**Ploting the most common ICU care units**

In [ ]:
# Visualise the most common ICU care-unit combinations
#
# Only the ten most frequent categories are displayed to keep
# the figure readable.

top_careunits = (
    icu_notes["first_careunits"]
    .value_counts()
    .head(10)
    .sort_values()
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_careunits.index,
    top_careunits.values
)

plt.title(
    "Ten Most Common ICU Care-Unit Categories"
)

plt.xlabel("Number of Discharge Summaries")
plt.ylabel("ICU Care Unit")

plt.tight_layout()
plt.show()

**Examine demographic and clinical characteristics**

In [ ]:
# Summarising demographic and clinical characteristics
#
# These statistics provide a basic description of the ICU
# discharge-summary cohort.

print("-" * 70)
print("ICU COHORT CHARACTERISTICS")
print("-" * 70)

print(f"Unique patients: {icu_notes['subject_id'].nunique():,}")

print(
    f"Median estimated age at admission: "
    f"{icu_notes['estimated_age_at_admission'].median():.1f} years"
)

print(
    f"Median hospital length of stay: "
    f"{icu_notes['hospital_los_days'].median():.2f} days"
)

print(
    f"Median total ICU length of stay: "
    f"{icu_notes['total_icu_los_days'].median():.2f} days"
)

print(
    f"In-hospital deaths: "
    f"{icu_notes['hospital_expire_flag'].sum():,}"
)

print("\nGender distribution:")
print(icu_notes["gender"].value_counts(dropna=False))

print("\nAdmission type distribution:")
print(icu_notes["admission_type"].value_counts(dropna=False))

**Displaying a complete discharge summary**

In [ ]:
# Displaying one complete ICU discharge summary
#
# MIMIC-IV notes are de-identified. However, the text should
# still be handled carefully and should not be copied into public
# documents unnecessarily.
#
# The first note is displayed only for research familiarisation.

sample_note = icu_notes.iloc[0]

print("-" * 70)
print("SAMPLE ICU DISCHARGE SUMMARY")
print("-" * 70)

print(f"Note ID      : {sample_note['note_id']}")
print(f"Subject ID   : {sample_note['subject_id']}")
print(f"Admission ID : {sample_note['hadm_id']}")
print(f"Care unit    : {sample_note['first_careunits']}")
print(f"Word count   : {sample_note['word_count']:,}")

print("\nDISCHARGE SUMMARY TEXT")
print("-" * 70)

print(sample_note["text"])

**Saving the clean ICU cohort**

In [ ]:
# Saving the clean linked ICU discharge-summary cohort
#
# The dataset is saved as a compressed CSV file in the outputs
# directory. This avoids repeating the complete merge process
# every time the notebook is reopened.

OUTPUT_PATH = (
    "/content/drive/MyDrive/Dissertation/outputs/"
)

ICU_NOTES_OUTPUT_FILE = (
    OUTPUT_PATH
    + "icu_discharge_summary_cohort.csv.gz"
)

icu_notes.to_csv(
    ICU_NOTES_OUTPUT_FILE,
    index=False,
    compression="gzip"
)

print("-" * 70)
print("CLEAN ICU COHORT SAVED SUCCESSFULLY")
print("-" * 70)

print(ICU_NOTES_OUTPUT_FILE)

# **5. Initial Data Exploration**

# **6. Data Quality Assessment**

# **7. ICU Cohort Identification**